# Análisis de sensibilidad de la evaluación de políticas

Sección 5.5 de la tesis. Todas las variaciones se evalúan sobre las diligencias observadas (247 sitios con resultado), con puntajes fuera de municipio, la política aplicada dentro de cada municipio y, salvo que se indique, el 30% del presupuesto histórico, costos heterogéneos $c_i = 1 + d_i$ (distancia a vías en km) y el GLM como modelo de la política.

- **A. Presupuesto**: 10%, 30% y 50% del presupuesto histórico.
- **B. Costos**: homogéneos; $1 + d_i$ (base); $1 + d_i/2$ (menor peso del acceso); $1 + 2d_i$ (mayor peso); $1 + $ distancia a centros poblados en km.
- **C. Sensibilidad de detección**: $r_i = s_i q_i$ con $q_i = 1$ (base); $q_i = 0.7$ homogéneo; $q_i$ heterogéneo según la cobertura vegetal ($q_i = 0.9 - 0.4\,\text{rango percentil del NDVI}$); $q_i$ heterogéneo según la pendiente ($q_i = 0.9 - 0.4\,\text{rango percentil de la pendiente}$). Los valores son ilustrativos.
- **D. Regla de agregación de celdas a sitios**: covariables en el centroide (base); máximo y $1 - \prod_j (1 - \hat p_j)$ sobre las celdas de 30 m a menos de 100 m del centroide.
- **E. Escala de los puntajes de Maxent** (con fondo del territorio): salida log-log complementaria frente a salida bruta.

Para cada variación se reportan los hallazgos observados de cada política y, en los paneles C a E, la fracción de sitios del portafolio de mochila que cambia frente a la especificación base del panel.


In [ ]:
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import elapid
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sys.path.append('../src')
from politicas import POLITICAS, seleccion

PATH_PROCESSED  = Path("../data/Processed/")
PATH_MUNICIPIOS = "../data/Municipios_Diciembre_2025_shp/Munpio.shp"
PATH_NDVI       = Path("../data/landsat/landsat_ndvi/")
PATH_DEM        = Path("../data/dem/dem_srtm/")

FRACCION_BASE = 0.3
RADIO_SITIO_M = 100
LADO_CELDA_M = 30
POLITICAS_CUADRO = ['Sin información', 'Mayor probabilidad de hallazgo',
                    'Probabilidad de hallazgo por costo', 'Mochila']
COVARIABLES = ['ndvi', 'altitud', 'pendiente', 'dist_hidrografia', 'dist_vias',
               'dist_centros_poblados', 'dist_grupos_armados']

pred = pd.read_csv(PATH_PROCESSED / "predicciones_fuera_municipio.csv", dtype={'municipio': str})
sitios = pd.read_csv(PATH_PROCESSED / "sitios.csv", dtype={'municipio': str})
obs = (pred[pred['label'].notna()]
       .merge(sitios[['sitio', 'disposicion']], on='sitio', how='left')
       .reset_index(drop=True))
obs['costo_homogeneo'] = 1.0
obs['costo_base'] = 1 + obs['dist_vias'] / 1000
print(f"Sitios con resultado: {len(obs)}  |  hallazgos: {int(obs['label'].sum())}")


In [ ]:
# ============================================================
# EVALUACIÓN
# ============================================================
def evaluar(df, col_puntaje, col_costo, f=FRACCION_BASE):
    """Hallazgos observados de cada política y portafolios de la heurística y la mochila."""
    salida, portafolios = {}, {}
    for nombre in POLITICAS_CUADRO:
        a = seleccion(df, col_puntaje, nombre, col_costo, f)
        salida[nombre] = (a * df['label']).sum()
        portafolios[nombre] = a
    return salida, portafolios

def fraccion_distinta(a, a_base):
    """Fracción de sitios del portafolio base que no están en el nuevo (y viceversa), sobre el base."""
    return np.sum(np.abs(a - a_base)) / (2 * a_base.sum())

filas = []
def registrar(panel, variacion, df, col_puntaje, col_costo, f=FRACCION_BASE, a_base=None):
    h, a = evaluar(df, col_puntaje, col_costo, f)
    distintos = lambda n: fraccion_distinta(a[n], a_base[n]) if a_base is not None else np.nan
    filas.append({'panel': panel, 'variacion': variacion, **h,
                  'distintos_heuristica': distintos('Probabilidad de hallazgo por costo'),
                  'distintos_mochila': distintos('Mochila')})
    return a


In [ ]:
# ============================================================
# A. PRESUPUESTO  y  B. COSTOS
# ============================================================
for f in [0.1, 0.3, 0.5]:
    registrar('A. Presupuesto', f'{f:.0%} del presupuesto histórico', obs, 'GLM', 'costo_base', f)

obs['costo_medio'] = 1 + obs['dist_vias'] / 2000
obs['costo_doble'] = 1 + 2 * obs['dist_vias'] / 1000
obs['costo_poblados'] = 1 + obs['dist_centros_poblados'] / 1000
for nombre, col in [('Homogéneos', 'costo_homogeneo'), ('$1 + d_i$ (base)', 'costo_base'),
                    ('$1 + d_i/2$', 'costo_medio'), ('$1 + 2d_i$', 'costo_doble'),
                    ('$1 +$ distancia a centros poblados', 'costo_poblados')]:
    registrar('B. Costos', nombre, obs, 'GLM', col)


In [ ]:
# ============================================================
# C. SENSIBILIDAD DE DETECCIÓN q_i
# ============================================================
a_base = registrar('C. Detección', '$q_i = 1$ (base)', obs, 'GLM', 'costo_base')
obs['r_q07'] = obs['GLM'] * 0.7
obs['r_q_ndvi'] = obs['GLM'] * (0.9 - 0.4 * obs['ndvi'].rank(pct=True))
obs['r_q_pendiente'] = obs['GLM'] * (0.9 - 0.4 * obs['pendiente'].rank(pct=True))
registrar('C. Detección', '$q_i = 0.7$', obs, 'r_q07', 'costo_base', a_base=a_base)
registrar('C. Detección', '$q_i$ según cobertura vegetal', obs, 'r_q_ndvi', 'costo_base', a_base=a_base)
registrar('C. Detección', '$q_i$ según pendiente', obs, 'r_q_pendiente', 'costo_base', a_base=a_base)


In [ ]:
# ============================================================
# D. REGLA DE AGREGACIÓN DE CELDAS A SITIOS
# ============================================================
# Se extraen las covariables en las celdas de 30 m a menos de RADIO_SITIO_M del centroide de cada
# sitio, igual que para los centroides. El GLM se entrena fuera de municipio con los centroides
# de los sitios de entrenamiento y predice las celdas de los sitios del municipio excluido.
municipios = gpd.read_file(PATH_MUNICIPIOS)
municipios['MpCodigo'] = municipios['MpCodigo'].astype(str).str.zfill(5)
municipios = municipios.to_crs("EPSG:4326").set_index('MpCodigo')

paso = np.arange(-RADIO_SITIO_M, RADIO_SITIO_M + 1, LADO_CELDA_M)
desplazamientos = np.array([(dx, dy) for dx in paso for dy in paso if np.hypot(dx, dy) <= RADIO_SITIO_M])
print(f"Celdas por sitio: {len(desplazamientos)}")

def muestrear(src, xy):
    v = np.array([s[0] for s in src.sample(xy)], dtype=float)
    if src.nodata is not None:
        v[v == src.nodata] = np.nan
    return v

celdas = []
for codigo, g in obs.groupby('municipio'):
    bounds = municipios.geometry[codigo].bounds
    stack = np.load(PATH_PROCESSED / f"covariables_{codigo}.npy", mmap_mode='r')
    h, w = stack.shape[1], stack.shape[2]
    with rasterio.open(PATH_NDVI / f"ndvi_{codigo}.tif") as src_ndvi, \
         rasterio.open(PATH_DEM / f"dem_{codigo}.tif") as src_dem:
        for i, sitio in g.iterrows():
            lon = sitio['lon'] + desplazamientos[:, 0] / (111320 * np.cos(np.radians(sitio['lat'])))
            lat = sitio['lat'] + desplazamientos[:, 1] / 110540
            xy = list(zip(lon, lat))
            z = muestrear(src_dem, xy)
            z_e = muestrear(src_dem, [(a + 0.001, b) for a, b in xy])
            z_n = muestrear(src_dem, [(a, b + 0.001) for a, b in xy])
            pendiente = np.degrees(np.arctan(np.hypot((z_e - z) / (0.001 * 111320),
                                                      (z_n - z) / (0.001 * 111320))))
            col = np.clip(((lon - bounds[0]) / (bounds[2] - bounds[0]) * w).astype(int), 0, w - 1)
            row = np.clip(((bounds[3] - lat) / (bounds[3] - bounds[1]) * h).astype(int), 0, h - 1)
            covs = np.column_stack([muestrear(src_ndvi, xy), z, pendiente,
                                    *[stack[c][row, col] for c in range(3, 7)]])
            celdas.append(pd.DataFrame(np.where(np.isnan(covs), 0.0, covs), columns=COVARIABLES)
                          .assign(fila_sitio=i, municipio=codigo))
celdas = pd.concat(celdas, ignore_index=True)

celdas['p'] = np.nan
for mun in obs['municipio'].unique():
    tr = obs['municipio'] != mun
    glm = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
    glm.fit(obs.loc[tr, COVARIABLES].values, obs.loc[tr, 'label'].values)
    en_mun = celdas['municipio'] == mun
    celdas.loc[en_mun, 'p'] = glm.predict_proba(celdas.loc[en_mun, COVARIABLES].values)[:, 1]

agregado = celdas.groupby('fila_sitio')['p'].agg(
    maximo='max',
    al_menos_una=lambda p: 1 - np.prod(1 - p),
    # -sum log(1 - p) ordena los sitios igual que 1 - prod(1 - p), pero sin saturarse en 1
    log_al_menos_una=lambda p: -np.sum(np.log1p(-p)))
obs['GLM_maximo'] = agregado['maximo']
obs['GLM_al_menos_una'] = agregado['al_menos_una']
obs['GLM_orden_al_menos_una'] = agregado['log_al_menos_una']
print(obs[['GLM', 'GLM_maximo', 'GLM_al_menos_una']].describe().round(3).to_string())
print(f"Sitios con 1 - prod(1 - p) > 0.999: {(obs['GLM_al_menos_una'] > 0.999).mean():.0%}")

ETIQ_AL_MENOS_UNA = f'$1 - \\prod_j (1 - \\hat p_j)$ en {RADIO_SITIO_M} m'
# Costos homogéneos: solo importa el orden (Proposición de invariancia, parte i)
a_base = registrar('D. Agregación', 'Centroide (base), costos homogéneos', obs, 'GLM', 'costo_homogeneo')
registrar('D. Agregación', f'Máximo en {RADIO_SITIO_M} m, costos homogéneos', obs, 'GLM_maximo',
          'costo_homogeneo', a_base=a_base)
registrar('D. Agregación', f'{ETIQ_AL_MENOS_UNA}, costos homogéneos', obs, 'GLM_orden_al_menos_una',
          'costo_homogeneo', a_base=a_base)
# Costos heterogéneos: importa la escala. Como 1 - prod(1 - p) es 1 en todos los sitios, la regla no
# ordena los sitios; la política de mayor probabilidad no está definida y las demás se guían por el costo.
a_base = registrar('D. Agregación', 'Centroide (base), costos heterogéneos', obs, 'GLM', 'costo_base')
registrar('D. Agregación', f'Máximo en {RADIO_SITIO_M} m, costos heterogéneos', obs, 'GLM_maximo',
          'costo_base', a_base=a_base)
obs['constante'] = 1.0
registrar('D. Agregación', f'{ETIQ_AL_MENOS_UNA}, costos heterogéneos', obs, 'constante',
          'costo_base', a_base=a_base)
filas[-1]['Mayor probabilidad de hallazgo'] = np.nan


In [ ]:
# ============================================================
# E. ESCALA DE LOS PUNTAJES DE MAXENT (fondo del territorio)
# ============================================================
fondo = pd.read_csv(PATH_PROCESSED / "fondo_maxent.csv", dtype={'municipio': str})
for col in ['Maxent_cloglog', 'Maxent_raw']:
    obs[col] = np.nan
for mun in obs['municipio'].unique():
    tr = obs['municipio'] != mun
    X_pres = obs.loc[tr & (obs['label'] == 1), COVARIABLES].values
    X_fondo = fondo.loc[fondo['municipio'].isin(obs.loc[tr, 'municipio']), COVARIABLES].values
    modelo = elapid.MaxentModel(feature_types=['linear'], beta_multiplier=1.5,
                                transform='cloglog', scorer='roc_auc')
    modelo.fit(np.vstack([X_pres, X_fondo]), np.r_[np.ones(len(X_pres)), np.zeros(len(X_fondo))])
    te = obs['municipio'] == mun
    obs.loc[te, 'Maxent_cloglog'] = modelo.predict(obs.loc[te, COVARIABLES].values)
    modelo.transform = 'raw'
    obs.loc[te, 'Maxent_raw'] = modelo.predict(obs.loc[te, COVARIABLES].values)

dif = (obs['Maxent_cloglog'] - obs['Maxent (fondo)']).abs()
print(f"Diferencia con la evaluación de modelos: máx. {dif.max():.2e}, media {dif.mean():.2e}")
monotona = all(obs.loc[obs['municipio'] == m, ['Maxent_cloglog', 'Maxent_raw']].corr('spearman').iloc[0, 1] > 0.9999
               for m in obs['municipio'].unique() if (obs['municipio'] == m).sum() > 2)
print(f"Dentro de cada municipio, la salida bruta y la cloglog tienen el mismo orden: {monotona}")

for costo, etiqueta in [('costo_homogeneo', 'costos homogéneos'), ('costo_base', 'costos heterogéneos')]:
    a_base = registrar('E. Escala de Maxent', f'Log-log complementaria (base), {etiqueta}',
                       obs, 'Maxent_cloglog', costo)
    registrar('E. Escala de Maxent', f'Salida bruta, {etiqueta}', obs, 'Maxent_raw', costo, a_base=a_base)


In [ ]:
# ============================================================
# CUADRO
# ============================================================
df_sens = pd.DataFrame(filas)
df_sens.to_csv(PATH_PROCESSED / "sensibilidad.csv", index=False)
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 60)
print(df_sens.round(2).to_string(index=False))
